In [35]:
import pandas as pd

text=pd.read_csv(r"C:\Users\Manisha\Desktop\Jetlearn-python-rishika\ML-and-AI\sentiments.txt",sep=";",names=["Sentences","Feeling"])
print(text)
text.info()

                                               Sentences  Feeling
0                                i didnt feel humiliated  sadness
1      i can go from feeling so hopeless to so damned...  sadness
2       im grabbing a minute to post i feel greedy wrong    anger
3      i am ever feeling nostalgic about the fireplac...     love
4                                   i am feeling grouchy    anger
...                                                  ...      ...
15995  i just had a very brief time in the beanbag an...  sadness
15996  i am now turning and i feel pathetic that i am...  sadness
15997                     i feel strong and good overall      joy
15998  i feel like this was such a rude comment and i...    anger
15999  i know a lot but i feel so stupid because i ca...  sadness

[16000 rows x 2 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ---

In [36]:
print(text["Feeling"].value_counts())

Feeling
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64


In [37]:
text.replace({"joy":1,"love":1,"surprise":1,"anger":0,"fear":0,"sadness":0},inplace=True)
print(text["Feeling"].value_counts())


Feeling
0    8762
1    7238
Name: count, dtype: int64


C:\Users\Manisha\AppData\Local\Temp\ipykernel_10480\2602090434.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  text.replace({"joy":1,"love":1,"surprise":1,"anger":0,"fear":0,"sadness":0},inplace=True)


In [38]:
import re
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download('averaged_perceptron_tagger_eng')
from nltk.corpus import stopwords,wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
# import nltk
# print(nltk.__file__)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Manisha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Manisha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Manisha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [39]:
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [40]:
wln=WordNetLemmatizer()
def transform(text):
    database=[]
    stwds=set(stopwords.words("english"))
    for sentence in text:
        new=re.sub("[^a-zA-Z]"," ",sentence)
        new=new.lower()
        new=new.split()
        tag_word=pos_tag(new)
        lemma=[]
        for word,tag in tag_word:
            if word not in stwds:
                lemma.append(wln.lemmatize(word,get_wordnet_pos(tag)))
        new_sentence=" ".join(lemma)
        database.append(new_sentence)
    return database





In [41]:
transformed=transform(text["Sentences"])
print(transformed[0:10])
print(text["Sentences"].head(10))

['didnt feel humiliate', 'go feel hopeless damned hopeful around someone care awake', 'im grab minute post feel greedy wrong', 'ever feel nostalgic fireplace know still property', 'feel grouchy', 'ive feel little burdened lately wasnt sure', 'ive take milligram time recommended amount ive fall asleep lot faster also feel like funny', 'feel confuse life teenager jade year old man', 'petronas year feel petronas perform well make huge profit', 'feel romantic']
0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
5    ive been feeling a little burdened lately wasn...
6    ive been taking or milligrams or times recomme...
7    i feel as confused about life as a teenager or...
8    i have been with petronas for years i feel tha...
9                                  i feel r

In [32]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(ngram_range=(1,2))
X=cv.fit_transform(transformed)
y=text["Feeling"]

In [33]:
from sklearn.model_selection import train_test_split

Xtrain,Xtest,ytrain,ytest=train_test_split(X,y,train_size=0.75,random_state=54)

In [42]:
from sklearn.ensemble import RandomForestClassifier

classifier=RandomForestClassifier(n_estimators=100)

classifier.fit(Xtrain,ytrain)
predicted=classifier.predict(Xtest)

from sklearn.metrics import confusion_matrix,classification_report
cm=confusion_matrix(ytest,predicted)
cr=classification_report(ytest,predicted)
print(cm)
print(cr)

[[2079  136]
 [ 158 1627]]
              precision    recall  f1-score   support

           0       0.93      0.94      0.93      2215
           1       0.92      0.91      0.92      1785

    accuracy                           0.93      4000
   macro avg       0.93      0.93      0.93      4000
weighted avg       0.93      0.93      0.93      4000



In [43]:
def prediction(text):
    transformed=transform(text)
    vectorised=cv.transform(transformed)
    pred=classifier.predict(vectorised)
    print(pred)
    for i in pred:
        if i==0:
            print("Negative")
        else:
            print("Positive")

In [44]:
sentence=["I had a bad day","I passed my test"]
prediction(sentence)

[0 0]
Negative
Negative


In [ ]:
parameters={"n_estimators":[50,100,200],
            "max_depth":[5,10]}
from sklearn.model_selection import GridSearchCV
gscv=GridSearchCV(RandomForestClassifier(),parameters,n_jobs=-1)#cv=2
gscv.fit(Xtrain,ytrain)
print(gscv.best_params_)